In [1]:
# %%
import os
import yaml
import cv2
import glob

# 1. 建立 KITTI 的 YOLO 設定檔
dataset_path = './kitti_dataset' # 替換為實際路徑
kitti_yaml = {
    'path': dataset_path,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'Car', 1: 'Van', 2: 'Truck', 3: 'Pedestrian',
        4: 'Person_sitting', 5: 'Cyclist', 6: 'Tram', 7: 'Misc', 8: 'DontCare'
    }
}

os.makedirs('dataset_configs', exist_ok=True)
with open('dataset_configs/kitti.yaml', 'w') as f:
    yaml.dump(kitti_yaml, f, default_flow_style=False)
print("KITTI 設定檔已建立於 dataset_configs/kitti.yaml")

# 2. 定義與執行 KITTI 到 YOLO 格式的轉換
def convert_kitti_to_yolo(kitti_label_path, image_path, yolo_label_path, classes):
    os.makedirs(os.path.dirname(yolo_label_path), exist_ok=True)
    img = cv2.imread(image_path)
    if img is None:
        return
    img_h, img_w = img.shape[:2]

    with open(kitti_label_path, 'r') as f_in, open(yolo_label_path, 'w') as f_out:
        for line in f_in:
            parts = line.strip().split(' ')
            cls_name = parts[0]
            if cls_name not in classes:
                continue
            
            cls_id = classes[cls_name]
            xmin, ymin, xmax, ymax = map(float, parts[4:8])
            
            x_center = ((xmin + xmax) / 2) / img_w
            y_center = ((ymin + ymax) / 2) / img_h
            width = (xmax - xmin) / img_w
            height = (ymax - ymin) / img_h
            
            f_out.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

# 執行轉換 (假設您的標籤在 label_2，圖片在 image_2)
splits = ['train', 'val']
for split in splits:
    img_dir = os.path.join(dataset_path, f'images/{split}')
    label_dir = os.path.join(dataset_path, f'kitti_labels/{split}') # KITTI 原始標籤路徑
    yolo_label_dir = os.path.join(dataset_path, f'labels/{split}')
    
    if os.path.exists(label_dir):
        label_files = glob.glob(os.path.join(label_dir, '*.txt'))
        for label_file in label_files:
            base_name = os.path.basename(label_file).replace('.txt', '.png')
            img_file = os.path.join(img_dir, base_name)
            yolo_file = os.path.join(yolo_label_dir, os.path.basename(label_file))
            convert_kitti_to_yolo(label_file, img_file, yolo_file, kitti_yaml['names'])

KITTI 設定檔已建立於 dataset_configs/kitti.yaml


In [2]:
# %%
from ultralytics import YOLO

# 使用 Ultralytics 官方的 Nano 模型作為起點 (請依據您的版本替換為 yolov8n.pt, yolo11n.pt 等)
# Nano 模型本身已具備 Resource-constrained 特性
model = YOLO('yolov8n.pt') 

epochs = 100
batch_size = 32
img_size = 640

print("開始在 KITTI 資料集上訓練模型...")

# 執行訓練
results = model.train(
    data='dataset_configs/kitti.yaml',
    epochs=epochs,
    imgsz=img_size,
    batch=batch_size,
    device=0, # 使用 GPU
    optimizer='AdamW',
    lr0=0.001
)
print("訓練完成。")

開始在 KITTI 資料集上訓練模型...
Ultralytics 8.4.56 🚀 Python-3.10.12 torch-2.12.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_configs/kitti.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None, optimize=False, opt

KeyboardInterrupt: 

In [ ]:
# %%
# 載入訓練好的最佳權重
best_model_path = 'runs/detect/train/weights/best.pt'
model = YOLO(best_model_path)

print("將模型匯出為 TensorRT (FP16) 以達到最快推論速度...")
# 將模型匯出為 TensorRT 引擎，大幅提升部署時的 FPS (需安裝 tensorrt 相關套件)
# 若無 NVIDIA 顯示卡，可將 format 改為 'openvino' (Intel) 或 'onnx'
model.export(format='engine', half=True, device=0)

# 載入加速後的模型
trt_model_path = 'runs/detect/train/weights/best.engine'
optimized_model = YOLO(trt_model_path)
print("加速模型載入完成。")

In [ ]:
# %%
import time
import torch

# 建立假資料模擬攝影機輸入
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dummy_frame = torch.randn(1, 3, 640, 640).to(device)

# 預熱模型 (Warm-up) - 確保硬體達到最高時脈
print("預熱模型中...")
for _ in range(10):
    _ = optimized_model(dummy_frame, verbose=False)

# 測量延遲與 FPS
print("開始效能測試...")
start_time = time.time()
num_frames = 100

for _ in range(num_frames):
    _ = optimized_model(dummy_frame, verbose=False)

end_time = time.time()
total_time = end_time - start_time
fps = num_frames / total_time
latency = (total_time / num_frames) * 1000

print(f"評估結果:")
print(f"平均推論延遲 (Latency): {latency:.2f} ms")
print(f"每秒幀數 (FPS): {fps:.2f}")

if fps >= 10:
    print("符合提案中最低 10 FPS 之要求。")
else:
    print("未達最低 10 FPS 門檻。")